# Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [ ]:
appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')
print(appart_df.head())

# Features

In [ ]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
   # "prix_m2_ref",
    "total_carrez_surface",
    "number_of_lots",
    "season"
]

target = "valeur_fonciere_actualisee"

# Encode season to numeric values
season_mapping = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
appart_df["season"] = appart_df["season"].map(season_mapping)

X = appart_df[features]
y = appart_df[target]

print("Feature count:", X.shape[1])
print(X.head())
print(y.head())

# Make sets : trail and test and validate

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

# Ssave

In [ ]:
os.makedirs("data_apt", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_apt/appart_train.csv", index=False)
val_df.to_csv("data_apt/appart_val.csv", index=False)
test_df.to_csv("data_apt/appart_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


# Train model

In [ ]:
model_appart = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_appart.fit(X_train, y_train)

print("Model trained.")

Model trained.


# Evaluation

In [ ]:
y_val_pred = model_appart.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

In [ ]:
df_eval = pd.DataFrame({
    "prix_reel": y_val,
    "prix_predit": y_val_pred
})

df_eval["erreur_pct"] = abs(df_eval["prix_reel"] - df_eval["prix_predit"]) / df_eval["prix_reel"] * 100

print(df_eval.head())

print("Erreur moyenne (%) :", df_eval["erreur_pct"].mean())

In [ ]:

y_train_pred = model_appart.predict(X_train)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print("Train RMSE:", rmse_train)
print("Train MAE:", mae_train)
print("Train R2:", r2_train)

SAVE MODEL

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_appart, "models/apt_random_forest_model.pkl")

print("Model saved.")

Model saved.


In [ ]:
print("Number of features expected:", model_appart.n_features_in_)
print("Feature names:", features)

In [ ]:
y_test_pred = model_appart.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("Test RMSE:", rmse_test)
print("Test MAE:", mae_test)
print("Test R2:", r2_test)

In [ ]:
df_test_eval = pd.DataFrame({
    "prix_reel": y_test,
    "prix_predit": y_test_pred
})

df_test_eval["erreur_pct"] = abs(df_test_eval["prix_reel"] - df_test_eval["prix_predit"]) / df_test_eval["prix_reel"] * 100

print("Erreur moyenne test (%) :", df_test_eval["erreur_pct"].mean())